# Clean and Pipeline Setup


# Imports

In [2]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    ConfusionMatrixDisplay,
    confusion_matrix,
    mean_squared_error,
    pairwise_distances,
    normalized_mutual_info_score,
    balanced_accuracy_score,
    roc_auc_score,
    log_loss,
    roc_curve,
)
from sklearn.utils.class_weight import compute_sample_weight

import seaborn as sns
import statsmodels.api as sm
import xgboost as xgb

In [3]:
def load_data(file):
    df = pd.read_csv(file)
    return df

def eda(df):
    print(df.head())
    print(df.dtypes)
    print(df.shape)

train_df = load_data("train.csv")
test_df = load_data("test.csv")

eda(train_df)

         client_id  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  \
0  CC_0000F52C3717      50000    2          1         2   26      1      2   
1  CC_0008C4E44BB9     150000    2          1         2   26     -1     -1   
2  CC_000B4B1BEDF6     450000    2          2         1   38     -2     -2   
3  CC_000D492CE7AD     230000    2          2         2   30      0      0   
4  CC_000DCFA992C4     160000    2          2         2   34     -1     -1   

   PAY_3  PAY_4  ...  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  \
0      0      0  ...      39475      40187      40992      2200      1962   
1     -1      0  ...      18400       1527       1527         0     18600   
2     -2     -2  ...      17255      17515          0     15008      1200   
3      0      0  ...     108826     100862      92481      6000      6000   
4     -1     -1  ...      13780      12297      12752     24000      8000   

   PAY_AMT3  PAY_AMT4  PAY_AMT5  PAY_AMT6  default  
0      1562    

client_id: non-informative competition identifier.
LIMIT_BAL: granted credit in NT dollars.
SEX: source-coded sex category.
EDUCATION: source-coded education category.
MARRIAGE: source-coded marital-status category.
AGE: age in years.
PAY_0, PAY_2 … PAY_6: recent repayment-status history.
BILL_AMT1 … BILL_AMT6: monthly bill statement amounts in NT dollars.
PAY_AMT1 … PAY_AMT6: monthly previous-payment amounts in NT dollars.
default: training target; 1 = default, 0 = no default.

# Cleaning data
- Check for missing values
- Convert to reasonable data type
- Save all standardisation normalisation for 

In [4]:
def missing_report(df, name):
    report = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_proportion": df.isna().mean(),
    })
    print(f"{name}: {len(df)} rows, {int(df.isna().sum().sum())} total missing values")
    print(report.to_string())
    print()
    return report

train_missing = missing_report(train_df, "train")
test_missing = missing_report(test_df, "test")


train: 24000 rows, 0 total missing values
           missing_count  missing_proportion
client_id              0                 0.0
LIMIT_BAL              0                 0.0
SEX                    0                 0.0
EDUCATION              0                 0.0
MARRIAGE               0                 0.0
AGE                    0                 0.0
PAY_0                  0                 0.0
PAY_2                  0                 0.0
PAY_3                  0                 0.0
PAY_4                  0                 0.0
PAY_5                  0                 0.0
PAY_6                  0                 0.0
BILL_AMT1              0                 0.0
BILL_AMT2              0                 0.0
BILL_AMT3              0                 0.0
BILL_AMT4              0                 0.0
BILL_AMT5              0                 0.0
BILL_AMT6              0                 0.0
PAY_AMT1               0                 0.0
PAY_AMT2               0                 0.0
PAY_AMT3     

In [5]:

print(train_df["default"].value_counts())
print(train_df["PAY_0"].value_counts())
print(train_df["PAY_4"].value_counts())



default
0    18691
1     5309
Name: count, dtype: int64
PAY_0
 0    11739
-1     4561
 1     2944
-2     2241
 2     2147
 3      262
 4       56
 5       18
 8       15
 6       11
 7        6
Name: count, dtype: int64
PAY_4
 0    13096
-1     4547
-2     3538
 2     2541
 3      144
 4       53
 7       45
 5       28
 6        4
 1        2
 8        2
Name: count, dtype: int64


Note: imbalance in data, with roughly 3.5:1 ratio no default to 0.
- PAY_I = status/severity of payment. Usually represents how many months delay
- Need to feautre engineer based on this 
- 

In [6]:
def engineer_features(df):
    df = df.copy()
    pay_amt_cols = [f"PAY_AMT{i}" for i in range(1, 7)]
    bill_cols = [f"BILL_AMT{i}" for i in range(1, 7)]
    pay_cols = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

    df["total_pay"] = df[pay_amt_cols].sum(axis=1)
    df["total_bill"] = df[bill_cols].sum(axis=1)

    for col in bill_cols:
        df[f"credit_util_{col[-1]}"] = df[col] / df["LIMIT_BAL"]

    months = np.arange(1, 7)
    x_centered = months - months.mean()
    bill_values = df[bill_cols].to_numpy()
    df["bill_slope"] = (bill_values * x_centered).sum(axis=1) / (x_centered**2).sum()

    for i in range(1, 6):
        prev_col = f"BILL_AMT{i}"
        next_col = f"BILL_AMT{i + 1}"
        df[f"bill_abs_change_{i}_{i + 1}"] = df[next_col] - df[prev_col]
        df[f"bill_pct_change_{i}_{i + 1}"] = (
            (df[next_col] - df[prev_col]) / df[prev_col].replace(0, np.nan)
        )

    df["bill_abs_change_1_6"] = df["BILL_AMT6"] - df["BILL_AMT1"]
    df["bill_pct_change_1_6"] = (
        (df["BILL_AMT6"] - df["BILL_AMT1"]) / df["BILL_AMT1"].replace(0, np.nan)
    )

    df["max_delay"] = df[pay_cols].max(axis=1)
    df["num_months_delayed"] = (df[pay_cols] > 0).sum(axis=1)
    df["num_severe_delays"] = (df[pay_cols] >= 2).sum(axis=1)
    df["ever_delayed"] = (df[pay_cols] > 0).any(axis=1).astype(int)
    df["mean_pay_status"] = df[pay_cols].mean(axis=1)

    return df


train_df = engineer_features(train_df)
test_df = engineer_features(test_df)

print(f"train shape after feature engineering: {train_df.shape}")
print(f"test shape after feature engineering: {test_df.shape}")
train_df.filter(like="bill_").head()


train shape after feature engineering: (24000, 51)
test shape after feature engineering: (6000, 50)


,bill_slope,bill_abs_change_1_2,bill_pct_change_1_2,bill_abs_change_2_3,bill_pct_change_2_3,bill_abs_change_3_4,bill_pct_change_3_4,bill_abs_change_4_5,bill_pct_change_4_5,bill_abs_change_5_6,bill_pct_change_5_6,bill_abs_change_1_6,bill_pct_change_1_6
0,-330.742857,19,0.000450,-3367,-0.079738,616,0.015852,712,0.018037,805,0.020031,-1215,-0.028787
1,-5092.685714,-38052,-1.000000,18600,NaN,-200,-0.010753,-16873,-0.917011,0,0.000000,-36525,-0.959871
2,-1256.057143,4033,0.367572,-1118,-0.074508,3368,0.242529,260,0.015068,-17515,-1.000000,-10972,-1.000000
3,-4442.257143,3871,0.034439,2194,0.018869,-9641,-0.081381,-7964,-0.073181,-8381,-0.083094,-19921,-0.177230
4,-680.085714,-7116,-0.375356,-3924,-0.331363,5862,0.740338,-1483,-0.107620,455,0.037001,-6206,-0.327355


In [9]:
# Correlation analysis: Spearman rank correlation and NMI

def discretize_for_nmi(series, n_bins=10):
    series = series.fillna(series.median())
    if series.nunique() <= n_bins:
        return series.astype(int)
    return pd.qcut(series, q=n_bins, duplicates="drop", labels=False)


def nmi_matrix(df, columns, n_bins=10):
    discretized = pd.DataFrame(
        {col: discretize_for_nmi(df[col], n_bins=n_bins) for col in columns},
        index=df.index,
    )

    n = len(columns)
    nmi = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            score = normalized_mutual_info_score(
                discretized[columns[i]], discretized[columns[j]]
            )
            nmi[i, j] = score
            nmi[j, i] = score

    return pd.DataFrame(nmi, index=columns, columns=columns)


feature_cols = train_df.select_dtypes(include=np.number).columns.tolist()

spearman_corr = train_df[feature_cols].corr(method="spearman").abs()
nmi_corr = nmi_matrix(train_df, feature_cols)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print("Spearman correlation matrix")
print(spearman_corr["default"].sort_values(ascending=False))
print("\nNMI matrix")
print(nmi_corr["default"].sort_values(ascending=False))


Spearman correlation matrix
default                1.000000
num_severe_delays      0.390202
num_months_delayed     0.388466
ever_delayed           0.353592
max_delay              0.321376
PAY_0                  0.294178
mean_pay_status        0.258100
PAY_2                  0.217069
PAY_3                  0.198490
PAY_4                  0.175762
total_pay              0.167173
LIMIT_BAL              0.167026
PAY_5                  0.164239
PAY_AMT1               0.153009
PAY_AMT2               0.148619
PAY_6                  0.146258
PAY_AMT3               0.132164
PAY_AMT4               0.124024
PAY_AMT6               0.117384
PAY_AMT5               0.111672
credit_util_4          0.100873
credit_util_6          0.100061
credit_util_5          0.098311
credit_util_3          0.094592
credit_util_2          0.091165
credit_util_1          0.078565
bill_pct_change_1_6    0.075459
bill_abs_change_1_6    0.075456
bill_slope             0.072783
bill_pct_change_1_2    0.057378
bill_abs_cha

num_severe_delays      0.390202
num_months_delayed     0.388466
ever_delayed           0.353592
max_delay              0.321376

Major variables - also consider other factors such as the total_pay, limit balance.
PAY_0                  0.294178
mean_pay_status        0.258100
PAY_2                  0.217069
PAY_3                  0.198490
PAY_4                  0.175762
total_pay              0.167173
LIMIT_BAL              0.167026

High spearmean vs low NMI - consider logistic regression for probability prediction. 

